# FinScore v2.0.1 — PUDIM revisada

## Avaliação econômico-financeira estrutural, ponderação adaptativa por PCA e sensibilidade Monte Carlo

Esta versão foi revisada sob duas perspectivas: **solidez estatística** e **prudência de crédito**.

O notebook:

1. preserva integralmente as contas reportadas;
2. separa ausência, zero, erro de leitura e inferência contábil;
3. bloqueia o score quando a base contém inconsistências críticas;
4. calcula um FinScore estrutural com pesos fixos;
5. calcula um FinScore adaptativo com apenas 30% de participação dos pesos sugeridos pelo PCA;
6. apresenta cobertura informacional e diagnósticos do PCA;
7. executa Monte Carlo como **análise de sensibilidade**, não como previsão de inadimplência;
8. mantém o Serasa como evidência externa, sem média automática;
9. executa autotestes reprodutíveis.

> O FinScore mede capacidade econômico-financeira relativa. Não estima probabilidade de default e não substitui análise cadastral, garantias, risco setorial, governança ou julgamento do analista.


## 0. Como usar

1. Coloque a planilha na mesma pasta do notebook ou informe seu caminho em `CAMINHO_PLANILHA`.
2. Confirme a aba `lancamentos`, com três exercícios e as 21 contas primárias.
3. Execute as células na ordem.
4. Corrija toda ocorrência marcada como `CRITICA` antes de usar o score em decisão.
5. Preencha os campos Serasa somente quando houver consulta válida e datada.

A execução com inconsistências críticas não falha silenciosamente: o notebook gera o diagnóstico, classifica a base como **NÃO APTA** e não calcula o score.


In [ ]:
# CONFIGURAÇÃO — edite apenas este bloco
from datetime import datetime
from pathlib import Path

VERSAO_MODELO = "2.0.5"
CAMINHO_PLANILHA = Path("C:\\Users\\ferna\\Documents\\dev\\Finscore\\FinScore\\V. 2 (Pudim)\\dados_teste\\DADOS_V2_PUDIM_bp-dre4.xlsx")
ABA_DADOS = "lancamentos"

NUM_SIMULACOES = 1000
SEMENTE = 20260723
EXECUTAR_AUTOTESTES = True
EXPORTAR_EXCEL = True

# Evidência externa: não entra no PCA nem é combinada por média.
SERASA_SCORE = 700               # inteiro ou float entre 0 e 1.000
SERASA_DATA_CONSULTA = "2026-07-23"       # exemplo: "2026-07-23"
SERASA_RESTRICAO_GRAVE = False

DATA_HORA_PROCESSAMENTO = datetime.now()
ARQUIVO_SAIDA = Path(
    f"resultados_finscore_{VERSAO_MODELO}_{DATA_HORA_PROCESSAMENTO:%Y%m%d_%H%M}.xlsx"
)

print(f"Versão: {VERSAO_MODELO}")
print(f"Planilha: {CAMINHO_PLANILHA.resolve()}")
print(f"Aba: {ABA_DADOS}")


Versão: 2.0.1-PUDIM-REVISADA
Planilha: C:\Users\ferna\Documents\dev\Finscore\FinScore\V. 2 (Pudim)\dados_teste\DADOS_V2_PUDIM_bp-dre4.xlsx
Aba: lancamentos


In [18]:
if not CAMINHO_PLANILHA.exists():
    disponiveis = sorted(Path.cwd().glob("*.xlsx"))
    print("Arquivo não encontrado. Planilhas disponíveis nesta pasta:")
    for arquivo in disponiveis:
        print(" -", arquivo.name)
    raise FileNotFoundError(
        f"Ajuste CAMINHO_PLANILHA. Não foi encontrado: {CAMINHO_PLANILHA}"
    )


## 1. Ambiente, contas e premissas


In [ ]:
import math
import re
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PRIMARY = [
    "p_Caixa_Equivalentes", "p_Contas_Receber_Clientes", "p_Estoques",
    "p_Ativo_Circulante", "p_Imobilizado_Liquido", "p_Ativo_Total",
    "p_Fornecedores", "p_Obrigacoes_Tributarias_CP",
    "p_Obrigacoes_Trabalhistas_CP", "p_Passivo_Circulante",
    "p_Passivo_Nao_Circulante", "p_Emprestimos_Financiamentos_CP",
    "p_Emprestimos_Financiamentos_LP", "p_Patrimonio_Liquido",
    "r_Receita_Liquida", "r_CMV_CPV_CSV", "r_Resultado_Antes_IR_CSLL",
    "r_Lucro_Liquido", "r_Receitas_Financeiras", "r_Despesa_de_Impostos",
    "r_Despesas_Financeiras",
]

SIGNED_ACCOUNTS = {
    "p_Patrimonio_Liquido", "r_Resultado_Antes_IR_CSLL", "r_Lucro_Liquido",
}
STRICTLY_POSITIVE = {"p_Ativo_Total"}
NONNEGATIVE = set(PRIMARY) - SIGNED_ACCOUNTS

DELTA_MIN = 0.05
DELTA_MAX = 0.35
BALANCE_TOLERANCE = 0.01
MAX_ATTEMPT_FACTOR = 30
MIN_NUCLEUS_COVERAGE = 0.70
PCA_ADAPTIVE_SHARE = 0.30
TEMPORAL_WEIGHTS = np.array([0.20, 0.30, 0.50], dtype=float) # Pesos FinScore V1 Brigadeiro
NUCLEUS_WEIGHTS = {"EO": 0.45, "FP": 0.55} # Pesos FinScore adicionados V2 Pudim

# r_Despesas_Financeiras só pode representar juros quando a origem da conta
# tiver sido conferida. Se False, Cobertura de Juros ficará indisponível.
USAR_DESPESAS_FINANCEIRAS_COMO_PROXY_JUROS = True

FIXED_WEIGHTS = {
    "crescimento_receita": 0.10, "margem_bruta": 0.20,
    "margem_ebit": 0.30, "margem_liquida": 0.20,
    "giro_ativo": 0.20,
    "liquidez_corrente": 0.15, "liquidez_seca": 0.10,
    "endividamento_exigivel": 0.25, "divida_liquida_ativo": 0.20,
    "composicao_endividamento": 0.10, "cobertura_juros": 0.20,
}

NUCLEI = {
    "EO": ["crescimento_receita", "margem_bruta", "margem_ebit",
           "margem_liquida", "giro_ativo"],
    "FP": ["liquidez_corrente", "liquidez_seca", "endividamento_exigivel",
           "divida_liquida_ativo", "composicao_endividamento",
           "cobertura_juros"],
}

# Curvas normativas preliminares. Cada par é (valor do índice, nota 0-100).
ANCHORS = {
    "crescimento_receita": [(-0.30, 0), (0, 50), (0.10, 80), (0.25, 100), (0.50, 70)],
    "margem_bruta": [(-0.10, 0), (0, 10), (0.15, 50), (0.30, 80), (0.50, 100)],
    "margem_ebit": [(-0.20, 0), (0, 40), (0.10, 75), (0.20, 100)],
    "margem_liquida": [(-0.20, 0), (0, 40), (0.07, 75), (0.15, 100)],
    "giro_ativo": [(0, 0), (0.30, 30), (0.70, 65), (1.20, 90), (2, 100)],
    "liquidez_corrente": [(0, 0), (0.70, 10), (1, 50), (1.30, 75), (1.80, 100), (4, 90)],
    "liquidez_seca": [(0, 0), (0.50, 10), (0.80, 45), (1.10, 75), (1.50, 100), (3, 90)],
    "endividamento_exigivel": [(0, 100), (0.30, 90), (0.50, 65), (0.70, 30), (1, 0), (1.50, 0)],
    "divida_liquida_ativo": [(-0.30, 100), (0, 95), (0.20, 75), (0.40, 40), (0.70, 0)],
    "composicao_endividamento": [(0, 100), (0.30, 85), (0.50, 60), (0.75, 25), (1, 0)],
    "cobertura_juros": [(-2, 0), (0, 5), (1, 30), (2, 60), (4, 85), (8, 100)],
}

SENSITIVITY_THRESHOLDS = [125, 250, 500]

assert len(PRIMARY) == 21
assert np.isclose(TEMPORAL_WEIGHTS.sum(), 1.0)
assert np.isclose(sum(NUCLEUS_WEIGHTS.values()), 1.0)
for nucleo, colunas in NUCLEI.items():
    assert np.isclose(sum(FIXED_WEIGHTS[c] for c in colunas), 1.0)

print("Ambiente carregado e premissas estruturais verificadas.")


Ambiente carregado e premissas estruturais verificadas.


### Regras metodológicas

- As contas reportadas nunca são sobrescritas para “fechar” o balanço.
- Uma inferência só é admitida quando há exatamente uma parcela ausente, as demais são conhecidas e não há conflito entre subtotal e componentes.
- Falhas críticas bloqueiam o score.
- O primeiro exercício não recebe crescimento neutro: ele fica ausente porque não existe base anterior.
- O giro do ativo usa ativo médio nos exercícios 2 e 3.
- O PCA opera sobre notas já orientadas como “maior = melhor”.
- Os pesos adaptativos são 70% pesos estruturais e 30% pesos sugeridos pelo PCA.
- O valor prudencial é o menor entre o score estrutural e o adaptativo.
- O Monte Carlo perturba a trajetória observada e mede sensibilidade; seus percentis não são intervalos de confiança nem probabilidades de default.


## 2. Importação, qualidade e preparação da base


In [20]:
def safe_div(a: pd.Series, b: pd.Series) -> pd.Series:
    denominator = b.astype(float).where(b.astype(float).abs() > 1e-12)
    return a.astype(float).div(denominator).replace([np.inf, -np.inf], np.nan)


def _is_blank_accounting_value(value) -> bool:
    if pd.isna(value):
        return True
    return str(value).strip().lower() in {"", "-", "--", "n/a", "na", "nan", "none", "null"}


def parse_accounting_value(value) -> float:
    # Converte números e textos contábeis; devolve NaN quando não interpretável.
    if _is_blank_accounting_value(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value) if np.isfinite(value) else np.nan

    text = str(value).strip()
    negative_parentheses = text.startswith("(") and text.endswith(")")
    text = re.sub(r"[^0-9,\.\-+]", "", text)
    if not text or text in {"-", "+", ".", ","}:
        return np.nan

    if "," in text and "." in text:
        if text.rfind(",") > text.rfind("."):
            text = text.replace(".", "").replace(",", ".")
        else:
            text = text.replace(",", "")
    elif "," in text:
        parts = text.split(",")
        text = "".join(parts) if len(parts[-1]) == 3 and len(parts) > 1 else text.replace(",", ".")
    elif text.count(".") > 1:
        parts = text.split(".")
        text = "".join(parts) if len(parts[-1]) == 3 else "".join(parts[:-1]) + "." + parts[-1]

    try:
        number = float(text)
    except ValueError:
        return np.nan
    return -abs(number) if negative_parentheses else number


def load_raw_data(path: Path, sheet_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    source = pd.read_excel(path, sheet_name=sheet_name)
    missing_columns = [c for c in ["ano", *PRIMARY] if c not in source.columns]
    if missing_columns:
        raise ValueError(f"Colunas ausentes na aba {sheet_name}: {missing_columns}")

    source = source[["ano", *PRIMARY]].dropna(how="all", subset=PRIMARY).copy()
    if len(source) != 3:
        raise ValueError(f"A aba deve ter 3 exercícios preenchidos; encontrei {len(source)}.")

    year_numeric = pd.to_numeric(source["ano"], errors="coerce")
    invalid_year = year_numeric.isna() | (year_numeric <= 0) | (year_numeric % 1 != 0)
    if invalid_year.any():
        rows = (source.index[invalid_year] + 2).tolist()
        raise ValueError(f"Exercício/ano deve ser inteiro positivo. Linhas inválidas: {rows}")
    source["ano"] = year_numeric.astype(int)
    if source["ano"].duplicated().any():
        years = source.loc[source["ano"].duplicated(keep=False), "ano"].tolist()
        raise ValueError(f"Há exercícios duplicados: {years}")
    source = source.sort_values("ano").reset_index(drop=True)

    report = []
    for column in PRIMARY:
        original = source[column].copy()
        converted = original.map(parse_accounting_value)
        blank = original.map(_is_blank_accounting_value)
        invalid = converted.isna() & ~blank
        absent = converted.isna() & blank
        if invalid.any():
            report.append({
                "severidade": "CRITICA", "tipo": "erro_conversao", "conta": column,
                "exercicios": ", ".join(source.loc[invalid, "ano"].astype(str)),
                "detalhe": "Valor textual não pôde ser convertido; revisar a origem.",
                "bloqueia_score": True,
            })
        if absent.any():
            report.append({
                "severidade": "AVISO", "tipo": "ausencia", "conta": column,
                "exercicios": ", ".join(source.loc[absent, "ano"].astype(str)),
                "detalhe": "Informação ausente preservada como NaN; não equivale a zero.",
                "bloqueia_score": False,
            })
        source[column] = converted

    report_df = pd.DataFrame(
        report,
        columns=["severidade", "tipo", "conta", "exercicios", "detalhe", "bloqueia_score"],
    )
    return source, report_df


def _known_sum(row: pd.Series, columns: list[str]) -> float:
    values = row[columns]
    return float(values.sum(skipna=True)) if values.notna().any() else np.nan


def validate_and_prepare(
    raw: pd.DataFrame,
    import_report: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    # Preserva raw; infere somente lacunas inequivocamente reconciliáveis.
    analysis = raw.copy(deep=True)
    issues = import_report.to_dict("records")

    def add(severity, kind, account, year, detail, blocks):
        issues.append({
            "severidade": severity, "tipo": kind, "conta": account,
            "exercicios": str(year), "detalhe": detail, "bloqueia_score": bool(blocks),
        })

    years = raw["ano"].astype(int).tolist()
    if any(b - a != 1 for a, b in zip(years[:-1], years[1:])):
        add("AVISO", "serie_temporal", "ano", ", ".join(map(str, years)),
            "Os três exercícios não são consecutivos.", False)

    for i, row in raw.iterrows():
        year = int(row["ano"])

        for account in NONNEGATIVE:
            value = row[account]
            if pd.notna(value) and value < 0:
                add("CRITICA", "sinal_invalido", account, year,
                    "Conta definida como não negativa contém valor negativo.", True)
        for account in STRICTLY_POSITIVE:
            value = row[account]
            if pd.notna(value) and value <= 0:
                add("CRITICA", "valor_nao_positivo", account, year,
                    "Conta deve ser estritamente positiva.", True)

        ac_parts = ["p_Caixa_Equivalentes", "p_Contas_Receber_Clientes", "p_Estoques"]
        pc_parts = ["p_Fornecedores", "p_Obrigacoes_Tributarias_CP",
                    "p_Obrigacoes_Trabalhistas_CP", "p_Emprestimos_Financiamentos_CP"]

        component_conflict = False
        if pd.notna(row["p_Ativo_Circulante"]):
            known = _known_sum(row, ac_parts)
            tolerance = BALANCE_TOLERANCE * max(abs(row["p_Ativo_Circulante"]), 1.0)
            if pd.notna(known) and known - row["p_Ativo_Circulante"] > tolerance:
                component_conflict = True
                add("CRITICA", "subtotal_inferior_componentes", "p_Ativo_Circulante", year,
                    f"Subtotal é {known - row['p_Ativo_Circulante']:,.2f} menor que componentes conhecidos.",
                    True)

        if pd.notna(row["p_Passivo_Circulante"]):
            known = _known_sum(row, pc_parts)
            tolerance = BALANCE_TOLERANCE * max(abs(row["p_Passivo_Circulante"]), 1.0)
            if pd.notna(known) and known - row["p_Passivo_Circulante"] > tolerance:
                component_conflict = True
                add("CRITICA", "subtotal_inferior_componentes", "p_Passivo_Circulante", year,
                    f"Subtotal é {known - row['p_Passivo_Circulante']:,.2f} menor que componentes conhecidos.",
                    True)

        pnc = row["p_Passivo_Nao_Circulante"]
        debt_lp = row["p_Emprestimos_Financiamentos_LP"]
        if pd.notna(pnc) and pd.notna(debt_lp):
            tolerance = BALANCE_TOLERANCE * max(abs(pnc), 1.0)
            if debt_lp - pnc > tolerance:
                component_conflict = True
                add("CRITICA", "subtotal_inferior_componentes", "p_Passivo_Nao_Circulante", year,
                    f"PNC é {debt_lp - pnc:,.2f} menor que empréstimos/financiamentos de LP.", True)

        if row[["p_Ativo_Total", "p_Ativo_Circulante", "p_Imobilizado_Liquido"]].notna().all():
            minimum_asset = row["p_Ativo_Circulante"] + row["p_Imobilizado_Liquido"]
            tolerance = BALANCE_TOLERANCE * max(abs(row["p_Ativo_Total"]), 1.0)
            if minimum_asset - row["p_Ativo_Total"] > tolerance:
                component_conflict = True
                add("CRITICA", "ativo_total_inferior_componentes", "p_Ativo_Total", year,
                    f"Ativo total é {minimum_asset - row['p_Ativo_Total']:,.2f} menor que AC + Imobilizado.",
                    True)

        balance_cols = ["p_Ativo_Total", "p_Passivo_Circulante",
                        "p_Passivo_Nao_Circulante", "p_Patrimonio_Liquido"]
        absent = [c for c in balance_cols if pd.isna(row[c])]
        if len(absent) == 1 and not component_conflict:
            account = absent[0]
            at, pc, pnc, pl = (row[c] for c in balance_cols)
            inferred = {
                "p_Ativo_Total": pc + pnc + pl,
                "p_Passivo_Circulante": at - pnc - pl,
                "p_Passivo_Nao_Circulante": at - pc - pl,
                "p_Patrimonio_Liquido": at - pc - pnc,
            }[account]
            valid = np.isfinite(inferred)
            valid &= inferred > 0 if account == "p_Ativo_Total" else True
            valid &= inferred >= 0 if account in {"p_Passivo_Circulante", "p_Passivo_Nao_Circulante"} else True
            if valid:
                analysis.at[i, account] = float(inferred)
                add("INFO", "inferencia_identidade", account, year,
                    f"Valor analítico inferido pela identidade contábil: {inferred:,.2f}. "
                    "O dado reportado original permanece NaN.", False)

        check_row = analysis.loc[i]
        if check_row[balance_cols].notna().all():
            difference = (
                check_row["p_Ativo_Total"]
                - check_row["p_Passivo_Circulante"]
                - check_row["p_Passivo_Nao_Circulante"]
                - check_row["p_Patrimonio_Liquido"]
            )
            tolerance = BALANCE_TOLERANCE * max(abs(check_row["p_Ativo_Total"]), 1.0)
            if abs(difference) > tolerance:
                add("CRITICA", "balanco_nao_fecha", "p_Patrimonio_Liquido", year,
                    f"Diferença de fechamento: {difference:,.2f} "
                    f"({difference / check_row['p_Ativo_Total']:.2%} do ativo). "
                    "Nenhuma conta foi sobrescrita.", True)
        else:
            missing_balance = [c for c in balance_cols if pd.isna(check_row[c])]
            add("CRITICA", "balanco_incompleto", ", ".join(missing_balance), year,
                "A identidade Ativo = PC + PNC + PL não pode ser verificada.", True)

    issues_df = pd.DataFrame(
        issues,
        columns=["severidade", "tipo", "conta", "exercicios", "detalhe", "bloqueia_score"],
    )
    if not issues_df.empty:
        order = pd.Categorical(
            issues_df["severidade"], categories=["CRITICA", "AVISO", "INFO"], ordered=True
        )
        issues_df = issues_df.assign(_ordem=order).sort_values(
            ["_ordem", "exercicios", "conta"]
        ).drop(columns="_ordem").reset_index(drop=True)

    critical_count = int(issues_df["bloqueia_score"].sum()) if not issues_df.empty else 0
    status = {
        "apto_score": critical_count == 0,
        "status": "APTA" if critical_count == 0 else "NAO APTA",
        "ocorrencias_criticas": critical_count,
        "ocorrencias_aviso": int((issues_df["severidade"] == "AVISO").sum()) if not issues_df.empty else 0,
        "inferencia_segura": int((issues_df["tipo"] == "inferencia_identidade").sum()) if not issues_df.empty else 0,
    }
    return analysis, issues_df, status


In [21]:
df_contas_reportadas, df_relatorio_importacao = load_raw_data(
    CAMINHO_PLANILHA, ABA_DADOS
)
df_contas_analise, df_qualidade, status_qualidade = validate_and_prepare(
    df_contas_reportadas, df_relatorio_importacao
)

print("Contas reportadas — preservadas sem sobrescrita:")
display(df_contas_reportadas)
print("Status da qualidade:")
display(pd.DataFrame([status_qualidade]))
print("Ocorrências de qualidade e reconciliação:")
display(df_qualidade)

if not status_qualidade["apto_score"]:
    print(
        "DECISÃO DO GATE: base NÃO APTA. "
        "O score e a simulação serão ignorados até a correção das ocorrências críticas."
    )


Contas reportadas — preservadas sem sobrescrita:


,ano,p_Caixa_Equivalentes,p_Contas_Receber_Clientes,p_Estoques,p_Ativo_Circulante,p_Imobilizado_Liquido,p_Ativo_Total,p_Fornecedores,p_Obrigacoes_Tributarias_CP,p_Obrigacoes_Trabalhistas_CP,p_Passivo_Circulante,p_Passivo_Nao_Circulante,p_Emprestimos_Financiamentos_CP,p_Emprestimos_Financiamentos_LP,p_Patrimonio_Liquido,r_Receita_Liquida,r_CMV_CPV_CSV,r_Resultado_Antes_IR_CSLL,r_Lucro_Liquido,r_Receitas_Financeiras,r_Despesa_de_Impostos,r_Despesas_Financeiras
0,1,"725,788.6700","8,665,337.2400",0.0000,"10,061,764.1200",457.5300,"10,178,773.5200",618.7500,"942,030.0500","1,725,519.6100","5,258,385.4000","1,144,867.4000","887,919.2900",NaN,"4,920,388.1200","34,952,306.5000",NaN,"13,799,492.5700","9,768,495.0000","10,972.7900","5,645,884.5800","998,713.9100"
1,2,"896,040.6500","6,543,935.0000",0.0000,"8,716,303.9700",0.0000,"8,833,313.3700",0.0000,"1,364,203.4500","1,582,991.1600","3,220,520.1400","1,144,867.4000","266,045.5300",NaN,"4,467,925.8300","43,265,605.2000",NaN,"19,203,714.9000","14,211,992.0000","16,307.5000","4,991,722.7300","13,600.3300"
2,3,"2,679,999.7300","1,088,677.9300",0.0000,"4,844,724.0100",0.0000,"4,845,181.5400",0.0000,"1,601,236.0500","2,257,170.7500","3,587,160.8000",NaN,"1,362,819.1900",NaN,"113,153.3400","48,658,455.8100",NaN,"17,382,202.5000","11,736,317.0000","111,796.1000","4,030,997.5700","11,543.7800"


Status da qualidade:


,apto_score,status,ocorrencias_criticas,ocorrencias_aviso,inferencia_segura
0,False,NAO APTA,3,3,0


Ocorrências de qualidade e reconciliação:


,severidade,tipo,conta,exercicios,detalhe,bloqueia_score
0,CRITICA,balanco_nao_fecha,p_Patrimonio_Liquido,1,"Diferença de fechamento: -1,144,867.40 (-11.25% do ativo). Nenhuma conta foi sobrescrita.",True
1,CRITICA,subtotal_inferior_componentes,p_Passivo_Circulante,3,"Subtotal é 1,634,065.19 menor que componentes conhecidos.",True
2,CRITICA,balanco_incompleto,p_Passivo_Nao_Circulante,3,A identidade Ativo = PC + PNC + PL não pode ser verificada.,True
3,AVISO,ausencia,p_Emprestimos_Financiamentos_LP,"1, 2, 3",Informação ausente preservada como NaN; não equivale a zero.,False
4,AVISO,ausencia,r_CMV_CPV_CSV,"1, 2, 3",Informação ausente preservada como NaN; não equivale a zero.,False
5,AVISO,ausencia,p_Passivo_Nao_Circulante,3,Informação ausente preservada como NaN; não equivale a zero.,False


DECISÃO DO GATE: base NÃO APTA. O score e a simulação serão ignorados até a correção das ocorrências críticas.


## 3. Contas derivadas, índices e curvas de pontuação


In [22]:
def derive(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["d_Ativo_Nao_Circulante"] = x.p_Ativo_Total - x.p_Ativo_Circulante
    x["d_Outros_Ativos_Circulantes"] = (
        x.p_Ativo_Circulante - x.p_Caixa_Equivalentes
        - x.p_Contas_Receber_Clientes - x.p_Estoques
    )
    x["d_Outros_Ativos_Nao_Circulantes"] = (
        x.p_Ativo_Total - x.p_Ativo_Circulante - x.p_Imobilizado_Liquido
    )
    x["d_Passivo_Exigivel_Total"] = (
        x.p_Passivo_Circulante + x.p_Passivo_Nao_Circulante
    )
    x["d_Outras_Obrigacoes_CP"] = (
        x.p_Passivo_Circulante - x.p_Fornecedores
        - x.p_Obrigacoes_Tributarias_CP - x.p_Obrigacoes_Trabalhistas_CP
        - x.p_Emprestimos_Financiamentos_CP
    )
    x["d_Outras_Obrigacoes_LP"] = (
        x.p_Passivo_Nao_Circulante - x.p_Emprestimos_Financiamentos_LP
    )
    x["d_Divida_Financeira_Bruta"] = (
        x.p_Emprestimos_Financiamentos_CP + x.p_Emprestimos_Financiamentos_LP
    )
    x["d_Divida_Financeira_Liquida"] = (
        x.d_Divida_Financeira_Bruta - x.p_Caixa_Equivalentes
    )
    x["d_Capital_Circulante_Liquido"] = (
        x.p_Ativo_Circulante - x.p_Passivo_Circulante
    )
    x["d_Ativo_Circulante_Operacional_Simplificado"] = (
        x.p_Contas_Receber_Clientes + x.p_Estoques
    )
    x["d_Passivo_Circulante_Operacional_Simplificado"] = (
        x.p_Fornecedores + x.p_Obrigacoes_Tributarias_CP
        + x.p_Obrigacoes_Trabalhistas_CP
    )
    x["d_Necessidade_Capital_Giro_Simplificada"] = (
        x.d_Ativo_Circulante_Operacional_Simplificado
        - x.d_Passivo_Circulante_Operacional_Simplificado
    )
    x["d_Saldo_Tesouraria_Simplificado"] = (
        x.d_Capital_Circulante_Liquido
        - x.d_Necessidade_Capital_Giro_Simplificada
    )
    x["d_Lucro_Bruto"] = x.r_Receita_Liquida - x.r_CMV_CPV_CSV
    x["d_Resultado_Financeiro_Liquido"] = (
        x.r_Receitas_Financeiras - x.r_Despesas_Financeiras
    )
    x["d_EBIT"] = (
        x.r_Resultado_Antes_IR_CSLL
        + x.r_Despesas_Financeiras
        - x.r_Receitas_Financeiras
    )
    x["d_Resultado_Apos_Impostos"] = (
        x.r_Resultado_Antes_IR_CSLL - x.r_Despesa_de_Impostos
    )
    x["d_Outros_Efeitos_Pos_Tributacao"] = (
        x.d_Resultado_Apos_Impostos - x.r_Lucro_Liquido
    )
    x["d_IR_CSLL_Outros_Efeitos"] = (
        x.r_Resultado_Antes_IR_CSLL - x.r_Lucro_Liquido
    )
    x["d_Ativo_Medio"] = (
        x.p_Ativo_Total + x.p_Ativo_Total.shift(1)
    ) / 2
    x.loc[x.index[0], "d_Ativo_Medio"] = np.nan
    return x


def indices(x: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=x.index)
    out["crescimento_receita"] = x.r_Receita_Liquida.pct_change(fill_method=None)
    out.loc[out.index[0], "crescimento_receita"] = np.nan
    out["margem_bruta"] = safe_div(x.d_Lucro_Bruto, x.r_Receita_Liquida)
    out["margem_ebit"] = safe_div(x.d_EBIT, x.r_Receita_Liquida)
    out["margem_liquida"] = safe_div(x.r_Lucro_Liquido, x.r_Receita_Liquida)
    out["giro_ativo"] = safe_div(x.r_Receita_Liquida, x.d_Ativo_Medio)
    out["liquidez_corrente"] = safe_div(
        x.p_Ativo_Circulante, x.p_Passivo_Circulante
    )
    out["liquidez_seca"] = safe_div(
        x.p_Ativo_Circulante - x.p_Estoques, x.p_Passivo_Circulante
    )
    out["endividamento_exigivel"] = safe_div(
        x.d_Passivo_Exigivel_Total, x.p_Ativo_Total
    )
    out["divida_liquida_ativo"] = safe_div(
        x.d_Divida_Financeira_Liquida, x.p_Ativo_Total
    )
    out["composicao_endividamento"] = safe_div(
        x.p_Passivo_Circulante, x.d_Passivo_Exigivel_Total
    )

    if USAR_DESPESAS_FINANCEIRAS_COMO_PROXY_JUROS:
        coverage = safe_div(x.d_EBIT, x.r_Despesas_Financeiras)
        zero_interest = x.r_Despesas_Financeiras.eq(0)
        coverage = coverage.mask(zero_interest & x.d_EBIT.gt(0), ANCHORS["cobertura_juros"][-1][0])
        coverage = coverage.mask(zero_interest & x.d_EBIT.lt(0), ANCHORS["cobertura_juros"][0][0])
        out["cobertura_juros"] = coverage
    else:
        out["cobertura_juros"] = np.nan
    return out


def score_indices(ind: pd.DataFrame) -> pd.DataFrame:
    scores = pd.DataFrame(index=ind.index)
    for column, points in ANCHORS.items():
        xp, fp = zip(*points)
        values = ind[column].to_numpy(float)
        scores[column] = np.interp(values, xp, fp, left=fp[0], right=fp[-1])
        scores.loc[ind[column].isna(), column] = np.nan
    return scores


## 4. Score estrutural e score adaptativo por PCA


In [23]:
@dataclass
class PCAProfile:
    weights: pd.Series
    diagnostics: dict
    loadings: pd.DataFrame


def _normalized_fixed_weights(columns: list[str]) -> pd.Series:
    weights = pd.Series({c: FIXED_WEIGHTS[c] for c in columns}, dtype=float)
    return weights / weights.sum()


def pca_profile(scores: pd.DataFrame, columns: list[str]) -> PCAProfile:
    fixed = _normalized_fixed_weights(columns)
    counts = scores[columns].notna().sum()
    usable = [
        c for c in columns
        if counts[c] >= 2 and scores[c].dropna().std(ddof=0) > 1e-12
    ]

    if len(usable) < 2:
        return PCAProfile(
            weights=fixed,
            diagnostics={
                "status_pca": "fallback_pesos_fixos",
                "variaveis_ativas": len(usable),
                "componentes": 0,
                "variancia_pc1": np.nan,
                "variancia_pc2": np.nan,
                "participacao_adaptativa": 0.0,
                "n_efetivo_pesos": 1 / np.square(fixed).sum(),
            },
            loadings=pd.DataFrame(),
        )

    zdf = scores[usable].copy()
    zdf = zdf.fillna(zdf.median())
    z = zdf.to_numpy(float)
    sd = z.std(axis=0, ddof=0)
    z = (z - z.mean(axis=0)) / sd

    n_components = min(2, z.shape[0] - 1, z.shape[1])
    pca = PCA(n_components=n_components, svd_solver="full")
    pca.fit(z)

    # |carga| torna a importância invariante à troca algébrica de sinal.
    pure_local = (
        np.abs(pca.components_).T * pca.explained_variance_ratio_
    ).sum(axis=1)
    pure_local = pure_local / pure_local.sum()
    pure = pd.Series(0.0, index=columns)
    pure.loc[usable] = pure_local

    adaptive = (1 - PCA_ADAPTIVE_SHARE) * fixed + PCA_ADAPTIVE_SHARE * pure
    adaptive = adaptive / adaptive.sum()

    loadings = pd.DataFrame(
        pca.components_.T,
        index=usable,
        columns=[f"PC{i+1}" for i in range(n_components)],
    )
    loadings["importancia_pca_pura"] = pure.loc[usable]
    loadings["peso_fixo"] = fixed.loc[usable]
    loadings["peso_adaptativo_final"] = adaptive.loc[usable]

    variance = pca.explained_variance_ratio_
    diagnostics = {
        "status_pca": "estimado_com_encolhimento",
        "variaveis_ativas": len(usable),
        "componentes": n_components,
        "variancia_pc1": float(variance[0]) if len(variance) > 0 else np.nan,
        "variancia_pc2": float(variance[1]) if len(variance) > 1 else np.nan,
        "participacao_adaptativa": PCA_ADAPTIVE_SHARE,
        "n_efetivo_pesos": float(1 / np.square(adaptive).sum()),
    }
    return PCAProfile(adaptive, diagnostics, loadings)


def _annual_nucleus(
    score_table: pd.DataFrame,
    columns: list[str],
    weights: pd.Series,
) -> tuple[np.ndarray, np.ndarray]:
    fixed = _normalized_fixed_weights(columns)
    values = score_table[columns].to_numpy(float)
    annual_scores = []
    annual_coverage = []
    for row in values:
        valid = np.isfinite(row)
        coverage = float(fixed.to_numpy()[valid].sum())
        annual_coverage.append(coverage)
        if not valid.any():
            annual_scores.append(np.nan)
            continue
        local_weights = weights.to_numpy()[valid]
        annual_scores.append(float(np.dot(row[valid], local_weights / local_weights.sum())))
    return np.asarray(annual_scores), np.asarray(annual_coverage)


def calculate_scores(score_table: pd.DataFrame) -> tuple[dict, dict[str, PCAProfile]]:
    profiles = {n: pca_profile(score_table, cols) for n, cols in NUCLEI.items()}
    result = {}

    for method in ("estrutural", "adaptativo"):
        nucleus_values = {}
        nucleus_coverage = {}
        for nucleus, columns in NUCLEI.items():
            weights = (
                _normalized_fixed_weights(columns)
                if method == "estrutural"
                else profiles[nucleus].weights
            )
            annual, coverage = _annual_nucleus(score_table, columns, weights)
            valid_years = np.isfinite(annual)
            if not valid_years.any():
                nucleus_values[nucleus] = np.nan
                nucleus_coverage[nucleus] = 0.0
                continue

            tw = TEMPORAL_WEIGHTS[valid_years]
            tw = tw / tw.sum()
            coverage_value = float(np.dot(coverage[valid_years], tw))
            nucleus_coverage[nucleus] = coverage_value
            nucleus_values[nucleus] = (
                float(np.dot(annual[valid_years], tw))
                if coverage_value >= MIN_NUCLEUS_COVERAGE
                else np.nan
            )

            result[f"cobertura_{nucleus}_{method}"] = coverage_value
            result[f"nucleo_{nucleus}_{method}"] = (
                10 * nucleus_values[nucleus]
                if np.isfinite(nucleus_values[nucleus]) else np.nan
            )

        if all(np.isfinite(nucleus_values[n]) for n in NUCLEI):
            result[f"finscore_{method}"] = 10 * sum(
                NUCLEUS_WEIGHTS[n] * nucleus_values[n] for n in NUCLEI
            )
        else:
            result[f"finscore_{method}"] = np.nan

    structural = result["finscore_estrutural"]
    adaptive = result["finscore_adaptativo"]
    result["finscore_prudencial"] = (
        min(structural, adaptive)
        if np.isfinite(structural) and np.isfinite(adaptive)
        else np.nan
    )
    result["divergencia_modelos"] = (
        abs(structural - adaptive)
        if np.isfinite(structural) and np.isfinite(adaptive)
        else np.nan
    )
    return result, profiles


In [24]:
MODELO_APTO = status_qualidade["apto_score"]

df_contas_derivadas = pd.DataFrame()
df_indices_observados = pd.DataFrame()
df_notas_observadas = pd.DataFrame()
finscore_observado = {}
pca_observado = {}

if MODELO_APTO:
    df_contas_derivadas = derive(df_contas_analise)
    df_indices_observados = indices(df_contas_derivadas)
    df_notas_observadas = score_indices(df_indices_observados)
    finscore_observado, pca_observado = calculate_scores(df_notas_observadas)

    print("FinScore observado:")
    display(pd.DataFrame([finscore_observado]).round(2))
    print("Índices observados:")
    display(df_indices_observados.set_axis(df_contas_analise["ano"]).round(4))

    diagnostics_rows = []
    weight_rows = []
    for nucleus, profile in pca_observado.items():
        diagnostics_rows.append({"nucleo": nucleus, **profile.diagnostics})
        for indicator, weight in profile.weights.items():
            weight_rows.append({
                "nucleo": nucleus,
                "indicador": indicator,
                "peso_adaptativo": weight,
                "peso_fixo": _normalized_fixed_weights(NUCLEI[nucleus])[indicator],
            })
    df_diagnostico_pca = pd.DataFrame(diagnostics_rows)
    df_pesos_pca = pd.DataFrame(weight_rows)
    display(df_diagnostico_pca.round(4))
    display(df_pesos_pca.round(4))
else:
    df_diagnostico_pca = pd.DataFrame()
    df_pesos_pca = pd.DataFrame()
    print("FinScore observado não calculado: gate de qualidade reprovado.")


FinScore observado não calculado: gate de qualidade reprovado.


## 5. Monte Carlo de sensibilidade


In [25]:
def triangular_widths(df: pd.DataFrame) -> dict[str, float]:
    widths = {}
    for column in PRIMARY:
        values = df[column].to_numpy(float)
        valid_pairs = (
            np.isfinite(values[:-1])
            & np.isfinite(values[1:])
            & (np.abs(values[:-1]) > 1e-12)
        )
        changes = np.abs(np.diff(values)[valid_pairs] / values[:-1][valid_pairs])
        raw_width = float(np.max(changes)) if changes.size else DELTA_MIN
        widths[column] = float(np.clip(raw_width, DELTA_MIN, DELTA_MAX))
    return widths


def _sample_nonnegative(
    values: np.ndarray,
    delta: float,
    rng: np.random.Generator,
) -> np.ndarray:
    sampled = values.astype(float).copy()
    active = np.isfinite(values) & (values > 1e-12)
    if active.any():
        sampled[active] = rng.triangular(
            np.maximum(0, values[active] * (1 - delta)),
            values[active],
            values[active] * (1 + delta),
        )
    return sampled


def _sample_signed(
    values: np.ndarray,
    delta: float,
    rng: np.random.Generator,
) -> np.ndarray:
    sampled = values.astype(float).copy()
    finite = np.isfinite(values)
    if finite.any():
        scale = max(float(np.nanmax(np.abs(values[finite]))), 1.0)
        half = np.maximum(np.abs(values) * delta, scale * DELTA_MIN)
        sampled[finite] = rng.triangular(
            values[finite] - half[finite],
            values[finite],
            values[finite] + half[finite],
        )
    return sampled


def _rebuild_total(
    sim: pd.DataFrame,
    base: pd.DataFrame,
    total: str,
    parts: list[str],
    width: float,
    rng: np.random.Generator,
) -> None:
    all_parts_known = base[parts].notna().all(axis=1)
    total_known = base[total].notna()
    rebuild = all_parts_known & total_known

    sim[total] = np.nan
    if rebuild.any():
        residual = (
            base.loc[rebuild, total]
            - base.loc[rebuild, parts].sum(axis=1)
        ).to_numpy(float)
        if (residual < -1e-9).any():
            raise ValueError(f"Residual negativo em {total}; a base deveria ter sido bloqueada.")
        residual = np.maximum(residual, 0)
        residual_sim = _sample_nonnegative(residual, width, rng)
        sim.loc[rebuild, total] = (
            sim.loc[rebuild, parts].sum(axis=1).to_numpy(float)
            + residual_sim
        )

    fallback = total_known & ~rebuild
    if fallback.any():
        sim.loc[fallback, total] = _sample_nonnegative(
            base.loc[fallback, total].to_numpy(float), width, rng
        )


def simulate_trajectory(
    base: pd.DataFrame,
    widths: dict[str, float],
    rng: np.random.Generator,
) -> pd.DataFrame:
    sim = base.copy(deep=True)

    balance_elements = [
        "p_Caixa_Equivalentes", "p_Contas_Receber_Clientes", "p_Estoques",
        "p_Imobilizado_Liquido", "p_Fornecedores", "p_Obrigacoes_Tributarias_CP",
        "p_Obrigacoes_Trabalhistas_CP", "p_Emprestimos_Financiamentos_CP",
        "p_Emprestimos_Financiamentos_LP",
    ]
    for column in balance_elements:
        sim[column] = _sample_nonnegative(
            base[column].to_numpy(float), widths[column], rng
        )

    _rebuild_total(
        sim, base, "p_Ativo_Circulante",
        ["p_Caixa_Equivalentes", "p_Contas_Receber_Clientes", "p_Estoques"],
        widths["p_Ativo_Circulante"], rng,
    )
    _rebuild_total(
        sim, base, "p_Passivo_Circulante",
        ["p_Fornecedores", "p_Obrigacoes_Tributarias_CP",
         "p_Obrigacoes_Trabalhistas_CP", "p_Emprestimos_Financiamentos_CP"],
        widths["p_Passivo_Circulante"], rng,
    )
    _rebuild_total(
        sim, base, "p_Passivo_Nao_Circulante",
        ["p_Emprestimos_Financiamentos_LP"],
        widths["p_Passivo_Nao_Circulante"], rng,
    )
    _rebuild_total(
        sim, base, "p_Ativo_Total",
        ["p_Ativo_Circulante", "p_Imobilizado_Liquido"],
        widths["p_Ativo_Total"], rng,
    )

    complete_balance = sim[
        ["p_Ativo_Total", "p_Passivo_Circulante", "p_Passivo_Nao_Circulante"]
    ].notna().all(axis=1)
    sim["p_Patrimonio_Liquido"] = np.nan
    sim.loc[complete_balance, "p_Patrimonio_Liquido"] = (
        sim.loc[complete_balance, "p_Ativo_Total"]
        - sim.loc[complete_balance, "p_Passivo_Circulante"]
        - sim.loc[complete_balance, "p_Passivo_Nao_Circulante"]
    )

    # DRE: sorteia drivers e reconstrói as identidades.
    for column in [
        "r_Receita_Liquida", "r_CMV_CPV_CSV", "r_Receitas_Financeiras",
        "r_Despesa_de_Impostos", "r_Despesas_Financeiras",
    ]:
        sim[column] = _sample_nonnegative(
            base[column].to_numpy(float), widths[column], rng
        )

    base_derived = derive(base)
    ebit_width = max(
        widths["r_Resultado_Antes_IR_CSLL"],
        widths["r_Receitas_Financeiras"],
        widths["r_Despesas_Financeiras"],
    )
    other_width = max(
        widths["r_Resultado_Antes_IR_CSLL"],
        widths["r_Despesa_de_Impostos"],
        widths["r_Lucro_Liquido"],
    )
    ebit_sim = _sample_signed(
        base_derived["d_EBIT"].to_numpy(float), ebit_width, rng
    )
    other_sim = _sample_signed(
        base_derived["d_Outros_Efeitos_Pos_Tributacao"].to_numpy(float),
        other_width, rng,
    )
    sim["r_Resultado_Antes_IR_CSLL"] = (
        ebit_sim - sim["r_Despesas_Financeiras"] + sim["r_Receitas_Financeiras"]
    )
    sim["r_Lucro_Liquido"] = (
        sim["r_Resultado_Antes_IR_CSLL"]
        - sim["r_Despesa_de_Impostos"]
        - other_sim
    )
    return sim


def accounting_flags(x: pd.DataFrame) -> list[str]:
    flags = []
    tests = {
        "conta_nao_negativa_negativa": x[list(NONNEGATIVE)].lt(0).any(axis=1),
        "ativo_total_nao_positivo": x.p_Ativo_Total.le(0),
        "AC_menor_componentes": x.p_Ativo_Circulante.lt(
            x[["p_Caixa_Equivalentes", "p_Contas_Receber_Clientes", "p_Estoques"]].sum(axis=1)
        ),
        "AT_menor_AC_Imobilizado": x.p_Ativo_Total.lt(
            x.p_Ativo_Circulante + x.p_Imobilizado_Liquido
        ),
        "PC_menor_componentes": x.p_Passivo_Circulante.lt(
            x[["p_Fornecedores", "p_Obrigacoes_Tributarias_CP",
               "p_Obrigacoes_Trabalhistas_CP",
               "p_Emprestimos_Financiamentos_CP"]].sum(axis=1)
        ),
        "PNC_menor_emprestimos_LP": x.p_Passivo_Nao_Circulante.lt(
            x.p_Emprestimos_Financiamentos_LP
        ),
        "balanco_nao_fecha": (
            x.p_Ativo_Total - x.p_Passivo_Circulante
            - x.p_Passivo_Nao_Circulante - x.p_Patrimonio_Liquido
        ).abs().gt(BALANCE_TOLERANCE * x.p_Ativo_Total.abs().clip(lower=1)),
    }
    for name, failed in tests.items():
        if failed.fillna(False).any():
            flags.append(name)
    return flags


def _relative_shocks(base: pd.DataFrame, sim: pd.DataFrame) -> dict:
    latest = base.index[-1]
    shocks = {}
    for column in PRIMARY:
        base_value = base.at[latest, column]
        sim_value = sim.at[latest, column]
        key = f"choque_{column}"
        if pd.notna(base_value) and pd.notna(sim_value) and abs(base_value) > 1e-12:
            shocks[key] = float((sim_value - base_value) / abs(base_value))
        else:
            shocks[key] = np.nan
    return shocks


def run_sensitivity(
    base: pd.DataFrame,
    n: int,
    seed: int,
) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    widths = triangular_widths(base)
    rows, flag_counts, rejected = [], {}, 0
    attempts = 0

    while len(rows) < n and attempts < n * MAX_ATTEMPT_FACTOR:
        attempts += 1
        sim = simulate_trajectory(base, widths, rng)
        flags = accounting_flags(sim)
        if flags:
            rejected += 1
            for flag in flags:
                flag_counts[flag] = flag_counts.get(flag, 0) + 1
            continue

        result, _ = calculate_scores(score_indices(indices(derive(sim))))
        if not all(np.isfinite(result[k]) for k in [
            "finscore_estrutural", "finscore_adaptativo", "finscore_prudencial"
        ]):
            rejected += 1
            flag_counts["score_indefinido"] = flag_counts.get("score_indefinido", 0) + 1
            continue

        result.update(_relative_shocks(base, sim))
        result.update({"simulacao": len(rows) + 1, "tentativa": attempts})
        rows.append(result)

    if len(rows) < n:
        raise RuntimeError(
            f"Somente {len(rows)} de {n} cenários válidos após {attempts} tentativas."
        )

    diagnostics = {
        "amplitudes": widths,
        "flags": flag_counts,
        "tentativas": attempts,
        "rejeitados": rejected,
        "taxa_rejeicao": rejected / attempts,
    }
    return pd.DataFrame(rows), diagnostics


def descriptive(results: pd.DataFrame, observed: dict) -> pd.DataFrame:
    rows = []
    for method in ("estrutural", "adaptativo", "prudencial"):
        column = f"finscore_{method}"
        series = results[column].dropna()
        if series.nunique() > 1:
            hist, edges = np.histogram(series, bins="fd")
            mode = (edges[hist.argmax()] + edges[hist.argmax() + 1]) / 2
        else:
            mode = float(series.iloc[0])
        row = {
            "metodo": method,
            "observado": observed.get(column, np.nan),
            "media": series.mean(),
            "mediana": series.median(),
            "moda_estimada": mode,
            "desvio_padrao": series.std(ddof=1),
            "minimo": series.min(),
            "p05": series.quantile(0.05),
            "p10": series.quantile(0.10),
            "p25": series.quantile(0.25),
            "p75": series.quantile(0.75),
            "p90": series.quantile(0.90),
            "p95": series.quantile(0.95),
            "maximo": series.max(),
        }
        for threshold in SENSITIVITY_THRESHOLDS:
            row[f"freq_abaixo_{threshold}"] = float((series < threshold).mean())
        rows.append(row)
    return pd.DataFrame(rows)


def sensitivity_ranking(results: pd.DataFrame) -> pd.DataFrame:
    shock_columns = [c for c in results if c.startswith("choque_")]
    output = []
    for score_column in [
        "finscore_estrutural", "finscore_adaptativo", "finscore_prudencial"
    ]:
        score_rank = results[score_column].rank(method="average")
        for shock in shock_columns:
            valid = results[shock].notna()
            if valid.sum() < 10 or results.loc[valid, shock].nunique() < 2:
                continue
            shock_rank = results.loc[valid, shock].rank(method="average")
            corr = shock_rank.corr(score_rank.loc[valid])
            output.append({
                "score": score_column,
                "conta": shock.removeprefix("choque_"),
                "correlacao_spearman": corr,
                "impacto_absoluto": abs(corr),
            })
    if not output:
        return pd.DataFrame(
            columns=["score", "conta", "correlacao_spearman", "impacto_absoluto"]
        )
    return pd.DataFrame(output).sort_values(
        ["score", "impacto_absoluto"], ascending=[True, False]
    ).reset_index(drop=True)


In [26]:
df_simulacoes = pd.DataFrame()
df_resumo = pd.DataFrame()
df_sensibilidade = pd.DataFrame()
df_amplitudes = pd.DataFrame()
diagnosticos_simulacao = {}

if MODELO_APTO:
    if NUM_SIMULACOES < 100:
        raise ValueError("Use ao menos 100 simulações.")

    df_simulacoes, diagnosticos_simulacao = run_sensitivity(
        df_contas_analise, NUM_SIMULACOES, SEMENTE
    )
    df_resumo = descriptive(df_simulacoes, finscore_observado)
    df_sensibilidade = sensitivity_ranking(df_simulacoes)
    df_amplitudes = pd.DataFrame({
        "conta": list(diagnosticos_simulacao["amplitudes"]),
        "amplitude_percentual": [
            100 * diagnosticos_simulacao["amplitudes"][c]
            for c in diagnosticos_simulacao["amplitudes"]
        ],
    })

    print(f"Cenários válidos: {len(df_simulacoes):,}")
    print(f"Tentativas: {diagnosticos_simulacao['tentativas']:,}")
    print(
        f"Rejeitados: {diagnosticos_simulacao['rejeitados']:,} "
        f"({diagnosticos_simulacao['taxa_rejeicao']:.2%})"
    )
    display(df_resumo.round(4))
    print("Contas com maior associação monotônica ao score prudencial:")
    display(
        df_sensibilidade[
            df_sensibilidade["score"].eq("finscore_prudencial")
        ].head(10).round(4)
    )
else:
    print("Monte Carlo não executado: gate de qualidade reprovado.")


Monte Carlo não executado: gate de qualidade reprovado.


In [27]:
if MODELO_APTO:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    methods = ["estrutural", "adaptativo", "prudencial"]
    colors = ["#2E86AB", "#D97706", "#7A3E9D"]

    for ax, method, color in zip(axes, methods, colors):
        column = f"finscore_{method}"
        ax.hist(
            df_simulacoes[column], bins="fd", color=color,
            alpha=0.78, edgecolor="white"
        )
        ax.axvline(
            df_simulacoes[column].median(), color="black",
            linestyle="--", label="Mediana"
        )
        observed_value = finscore_observado.get(column, np.nan)
        if np.isfinite(observed_value):
            ax.axvline(
                observed_value, color="#B91C1C",
                linestyle=":", label="Observado"
            )
        ax.set_title(method.upper())
        ax.set_xlabel("FinScore")
        ax.legend()

    axes[0].set_ylabel("Frequência")
    fig.suptitle("Distribuições de sensibilidade — não são probabilidades de default")
    plt.tight_layout()
    plt.show()


## 6. Evidência externa — Serasa


In [28]:
def assess_external_credit(
    finscore_prudential,
    serasa_score,
    consultation_date=None,
    severe_restriction=False,
) -> pd.DataFrame:
    if serasa_score is None or pd.isna(serasa_score):
        return pd.DataFrame([{
            "serasa_score": np.nan,
            "data_consulta": consultation_date,
            "status": "AUSENTE",
            "divergencia_pontos": np.nan,
            "nivel_divergencia": "não calculado",
            "direcao": "avaliação externa ausente",
            "restricao_grave": bool(severe_restriction),
            "score_integrado": np.nan,
        }])
    if not 0 <= float(serasa_score) <= 1000:
        raise ValueError("SERASA_SCORE deve estar entre 0 e 1.000.")

    if pd.isna(finscore_prudential):
        difference = np.nan
        level = "não calculado"
        direction = "FinScore indisponível"
    else:
        signed = float(serasa_score) - float(finscore_prudential)
        difference = abs(signed)
        if difference <= 100:
            level = "baixa"
        elif difference <= 200:
            level = "moderada"
        elif difference <= 300:
            level = "relevante"
        else:
            level = "elevada"
        direction = (
            "convergente" if difference <= 100
            else "evidência externa mais favorável" if signed > 0
            else "evidência externa mais desfavorável"
        )

    status = "ESCALONAR_RESTRICAO_GRAVE" if severe_restriction else "ANALISAR_CONJUNTAMENTE"
    return pd.DataFrame([{
        "serasa_score": float(serasa_score),
        "data_consulta": consultation_date,
        "status": status,
        "divergencia_pontos": difference,
        "nivel_divergencia": level,
        "direcao": direction,
        "restricao_grave": bool(severe_restriction),
        # Deliberadamente ausente: não há média automática.
        "score_integrado": np.nan,
    }])


df_serasa = assess_external_credit(
    finscore_observado.get("finscore_prudencial", np.nan),
    SERASA_SCORE,
    SERASA_DATA_CONSULTA,
    SERASA_RESTRICAO_GRAVE,
)
display(df_serasa)


,serasa_score,data_consulta,status,divergencia_pontos,nivel_divergencia,direcao,restricao_grave,score_integrado
0,NaN,None,AUSENTE,NaN,não calculado,avaliação externa ausente,False,NaN


## 7. Autotestes


In [29]:
def synthetic_valid_data() -> pd.DataFrame:
    data = {
        "ano": [2023, 2024, 2025],
        "p_Caixa_Equivalentes": [100, 120, 150],
        "p_Contas_Receber_Clientes": [300, 330, 360],
        "p_Estoques": [200, 210, 220],
        "p_Ativo_Circulante": [700, 760, 830],
        "p_Imobilizado_Liquido": [500, 540, 580],
        "p_Ativo_Total": [1400, 1500, 1600],
        "p_Fornecedores": [150, 160, 170],
        "p_Obrigacoes_Tributarias_CP": [40, 45, 50],
        "p_Obrigacoes_Trabalhistas_CP": [60, 65, 70],
        "p_Passivo_Circulante": [400, 420, 430],
        "p_Passivo_Nao_Circulante": [300, 280, 250],
        "p_Emprestimos_Financiamentos_CP": [100, 90, 80],
        "p_Emprestimos_Financiamentos_LP": [250, 230, 200],
        "p_Patrimonio_Liquido": [700, 800, 920],
        "r_Receita_Liquida": [2000, 2200, 2420],
        "r_CMV_CPV_CSV": [1200, 1300, 1400],
        "r_Resultado_Antes_IR_CSLL": [200, 240, 300],
        "r_Lucro_Liquido": [140, 168, 210],
        "r_Receitas_Financeiras": [10, 10, 12],
        "r_Despesa_de_Impostos": [60, 72, 90],
        "r_Despesas_Financeiras": [50, 45, 40],
    }
    return pd.DataFrame(data)


def run_self_tests() -> pd.DataFrame:
    tests = []

    def check(name, condition, detail=""):
        tests.append({
            "teste": name,
            "status": "PASSOU" if bool(condition) else "FALHOU",
            "detalhe": detail,
        })

    base = synthetic_valid_data()
    original = base.copy(deep=True)
    empty_report = pd.DataFrame(
        columns=["severidade", "tipo", "conta", "exercicios", "detalhe", "bloqueia_score"]
    )
    prepared, quality, status = validate_and_prepare(base, empty_report)

    check("21 contas primárias", len(PRIMARY) == 21)
    check("base sintética aprovada", status["apto_score"])
    check("dados reportados imutáveis", base.equals(original))

    derived = derive(prepared)
    balance_difference = (
        prepared.p_Ativo_Total - prepared.p_Passivo_Circulante
        - prepared.p_Passivo_Nao_Circulante - prepared.p_Patrimonio_Liquido
    )
    check("identidade do balanço", np.allclose(balance_difference, 0))

    index_table = indices(derived)
    check("crescimento inicial ausente", pd.isna(index_table.iloc[0]["crescimento_receita"]))
    score_table = score_indices(index_table)
    scores, profiles = calculate_scores(score_table)
    check(
        "scores limitados a 0-1000",
        all(0 <= scores[k] <= 1000 for k in [
            "finscore_estrutural", "finscore_adaptativo", "finscore_prudencial"
        ]),
    )
    check(
        "prudencial é o menor",
        np.isclose(
            scores["finscore_prudencial"],
            min(scores["finscore_estrutural"], scores["finscore_adaptativo"]),
        ),
    )
    check(
        "pesos PCA válidos",
        all(
            np.isclose(profile.weights.sum(), 1.0)
            and (profile.weights >= 0).all()
            for profile in profiles.values()
        ),
    )

    result_a, diagnostics_a = run_sensitivity(prepared, 120, 12345)
    result_b, diagnostics_b = run_sensitivity(prepared, 120, 12345)
    score_columns = [
        "finscore_estrutural", "finscore_adaptativo", "finscore_prudencial"
    ]
    check(
        "reprodutibilidade por semente",
        np.allclose(result_a[score_columns], result_b[score_columns]),
    )
    check(
        "simulações dentro da escala",
        result_a[score_columns].ge(0).all().all()
        and result_a[score_columns].le(1000).all().all(),
    )

    corrupted = base.copy()
    corrupted.loc[2, "p_Passivo_Circulante"] = 100
    _, corrupt_quality, corrupt_status = validate_and_prepare(corrupted, empty_report)
    check(
        "gate bloqueia subtotal inconsistente",
        (not corrupt_status["apto_score"])
        and corrupt_quality["tipo"].eq("subtotal_inferior_componentes").any(),
    )

    serasa_test = assess_external_credit(scores["finscore_prudencial"], 500)
    check(
        "Serasa não gera média automática",
        pd.isna(serasa_test.loc[0, "score_integrado"]),
    )

    return pd.DataFrame(tests)


df_autotestes = run_self_tests() if EXECUTAR_AUTOTESTES else pd.DataFrame()
if EXECUTAR_AUTOTESTES:
    display(df_autotestes)
    if not df_autotestes["status"].eq("PASSOU").all():
        raise AssertionError("Há autoteste(s) com falha; revise antes de usar o modelo.")


,teste,status,detalhe
0,21 contas primárias,PASSOU,
1,base sintética aprovada,PASSOU,
2,dados reportados imutáveis,PASSOU,
3,identidade do balanço,PASSOU,
4,crescimento inicial ausente,PASSOU,
5,scores limitados a 0-1000,PASSOU,
6,prudencial é o menor,PASSOU,
7,pesos PCA válidos,PASSOU,
8,reprodutibilidade por semente,PASSOU,
9,simulações dentro da escala,PASSOU,


## 8. Exportação opcional


In [30]:
df_configuracao = pd.DataFrame({
    "parametro": [
        "versao", "planilha", "aba", "simulacoes", "semente",
        "data_hora_processamento", "delta_min", "delta_max",
        "peso_EO", "peso_FP", "participacao_pca",
        "cobertura_minima_nucleo", "proxy_juros", "status_base",
    ],
    "valor": [
        VERSAO_MODELO, str(CAMINHO_PLANILHA), ABA_DADOS, NUM_SIMULACOES,
        SEMENTE, DATA_HORA_PROCESSAMENTO.strftime("%Y-%m-%d %H:%M:%S"),
        DELTA_MIN, DELTA_MAX, NUCLEUS_WEIGHTS["EO"], NUCLEUS_WEIGHTS["FP"],
        PCA_ADAPTIVE_SHARE, MIN_NUCLEUS_COVERAGE,
        USAR_DESPESAS_FINANCEIRAS_COMO_PROXY_JUROS,
        status_qualidade["status"],
    ],
})

if EXPORTAR_EXCEL:
    with pd.ExcelWriter(ARQUIVO_SAIDA, engine="openpyxl") as writer:
        pd.DataFrame([status_qualidade]).to_excel(
            writer, sheet_name="status_modelo", index=False
        )
        df_qualidade.to_excel(writer, sheet_name="qualidade_dados", index=False)
        df_contas_reportadas.to_excel(
            writer, sheet_name="contas_reportadas", index=False
        )
        df_contas_analise.to_excel(
            writer, sheet_name="contas_analise", index=False
        )
        df_serasa.to_excel(writer, sheet_name="evidencia_serasa", index=False)
        df_autotestes.to_excel(writer, sheet_name="autotestes", index=False)
        df_configuracao.to_excel(writer, sheet_name="configuracao", index=False)

        if MODELO_APTO:
            pd.DataFrame([finscore_observado]).to_excel(
                writer, sheet_name="score_observado", index=False
            )
            df_contas_derivadas.to_excel(
                writer, sheet_name="contas_derivadas", index=False
            )
            df_indices_observados.to_excel(
                writer, sheet_name="indices_observados", index=False
            )
            df_notas_observadas.to_excel(
                writer, sheet_name="notas_observadas", index=False
            )
            df_diagnostico_pca.to_excel(
                writer, sheet_name="diagnostico_pca", index=False
            )
            df_pesos_pca.to_excel(writer, sheet_name="pesos_pca", index=False)
            df_resumo.to_excel(writer, sheet_name="resumo_simulacao", index=False)
            df_simulacoes.to_excel(writer, sheet_name="simulacoes", index=False)
            df_sensibilidade.to_excel(
                writer, sheet_name="sensibilidade", index=False
            )
            df_amplitudes.to_excel(writer, sheet_name="amplitudes", index=False)

    print(f"Resultados salvos em: {ARQUIVO_SAIDA.resolve()}")
else:
    print("Exportação Excel desativada. Defina EXPORTAR_EXCEL = True para gerar o arquivo.")


Resultados salvos em: C:\Users\ferna\Documents\dev\Finscore\FinScore\V. 2 (Pudim)\algoritmos\resultados_finscore_2.0.1-PUDIM-REVISADA_20260723_1309.xlsx


## 9. Interpretação e limites

- **Score estrutural:** régua principal, baseada em pesos econômicos fixos e auditáveis.
- **Score adaptativo:** preserva 70% da régua estrutural e permite ao PCA ajustar 30% dos pesos conforme a trajetória da empresa.
- **Score prudencial:** menor valor entre estrutural e adaptativo; é uma regra de decisão conservadora, não uma estimativa estatística de perda.
- **Cobertura:** impede que a ausência de indicadores relevantes seja ocultada por reponderação excessiva.
- **PCA:** identifica padrões de variância nas notas; não demonstra causalidade, relevância econômica ou previsão de default.
- **Monte Carlo:** mede sensibilidade às amplitudes triangulares e à reestimação dos pesos. Os percentis são distribuições condicionais às premissas do código.
- **Serasa:** permanece separado. Divergência informa necessidade de investigação; não significa erro de um dos scores.
- **Gate de qualidade:** tem precedência sobre qualquer score. Uma demonstração inconsistente não se torna confiável porque o modelo consegue produzir um número.

### Referências metodológicas

- Federal Reserve, OCC e FDIC — Revised Guidance on Model Risk Management (SR 26-2, 2026):  
  https://www.federalreserve.gov/supervisionreg/srletters/SR2602.htm
- Basel Committee — Principles for effective risk data aggregation and risk reporting:  
  https://www.bis.org/publ/bcbs239.pdf
- Basel Committee — Minimum requirements for the IRB approach:  
  https://www.bis.org/basel_framework/chapter/CRE/36.htm
- Scikit-learn — documentação oficial de PCA:  
  https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
